# LFM2.5-Audio × vLLM-Omni — itération GPU (Colab)

Poste de travail pour valider le plugin out-of-tree `lfm2_audio.vllm_plugin`
(repo [`rcarvalo/finetuning_s2s_toolcalling`](https://github.com/rcarvalo/finetuning_s2s_toolcalling), branche `claude/blissful-tesla-7i1yky`).

Boucle d'itération : éditer en local → push → **cellule 2** (pull) → **cellule 4** (smoke) → lire le diagnostic → recommencer.

Étapes du smoke (`scripts/colab_smoke_vllm_omni.py`) : `imports` → `plugin` → `contract` → `checkpoint` → `engine`.
Critère bloquant P2 : parité greedy avec `liquid_audio.generate_interleaved` (`tests/test_omni_parity.py`).

In [ ]:
# 1. GPU
!nvidia-smi

In [ ]:
# 2. Repo (clone idempotent + pull)
import os
BRANCH = "claude/blissful-tesla-7i1yky"
REPO = "https://github.com/rcarvalo/finetuning_s2s_toolcalling.git"
if not os.path.isdir("/content/finetuning_s2s_toolcalling"):
    !git clone -b {BRANCH} {REPO} /content/finetuning_s2s_toolcalling
%cd /content/finetuning_s2s_toolcalling
!git pull --ff-only
!git log --oneline -3

In [ ]:
# 3. Dépendances.
#    vllm-omni ne déclare pas vllm (versions appariées major.minor) et le build
#    PyPI de vllm 0.22 est CUDA 13 (libcudart.so.13, absent de Colab) →
#    wheel officiel +cu129 du release GitHub + torch assorti (index cu129).
import subprocess, sys
VLLM_WHL = "https://github.com/vllm-project/vllm/releases/download/v0.22.1/vllm-0.22.1+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
TORCH_IDX = "https://download.pytorch.org/whl/cu129"
!{sys.executable} -m pip install -q "vllm @ {VLLM_WHL}" --extra-index-url {TORCH_IDX}

# Garde-fou : si une tentative précédente a laissé un torch CUDA 13, on le réaligne.
cuda = subprocess.run([sys.executable, "-c", "import torch; print(torch.version.cuda)"],
                      capture_output=True, text=True).stdout.strip()
print("torch CUDA:", cuda or "?")
if cuda.startswith("13"):
    tv = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__.split('+')[0])"],
                        capture_output=True, text=True).stdout.strip()
    !{sys.executable} -m pip install -q --force-reinstall --no-deps "torch=={tv}+cu129" --index-url {TORCH_IDX}
    print("⚠️ torch réaligné sur cu129 — Exécution → Redémarrer la session, puis reprendre ici.")

!{sys.executable} -m pip install -q "vllm-omni==0.22.0" "liquid-audio>=1.3.0"
!{sys.executable} -m pip install -q -e . --no-deps
import importlib.metadata as md
print("vllm", md.version("vllm"), "| vllm-omni", md.version("vllm-omni"),
      "| liquid-audio", md.version("liquid-audio"))


In [ ]:
# 4. Smoke progressif (sans checkpoint : imports + plugin + contrat runtime)
import sys
!{sys.executable} scripts/colab_smoke_vllm_omni.py

In [ ]:
# 5. Checkpoint : base LFM2.5-Audio convertie au layout vLLM-Omni
#    (suffisant pour valider chargement/engine ; le finetuné FR viendra après)
import sys
from huggingface_hub import snapshot_download
base = snapshot_download("LiquidAI/LFM2.5-Audio-1.5B")
!{sys.executable} -m lfm2_audio.vllm_plugin.convert_checkpoint --checkpoint {base} --output /content/lfm25_audio_omni
!{sys.executable} scripts/colab_smoke_vllm_omni.py --checkpoint /content/lfm25_audio_omni

In [ ]:
# 6. Engine (le gros test : Omni(...) démarre et génère)
import sys
!{sys.executable} scripts/colab_smoke_vllm_omni.py --checkpoint /content/lfm25_audio_omni --engine

In [ ]:
# 7. Parité P2 (bloquant) — une fois l'engine fonctionnel
import sys
!OMNI_CHECKPOINT=/content/lfm25_audio_omni BASE_MODEL=LiquidAI/LFM2.5-Audio-1.5B \
  {sys.executable} -m pytest tests/test_omni_parity.py -m gpu -q -x